# EvoPrompt iter2: standard deviation table for before/after metrics

Этот ноутбук считает **стандартное отклонение** по тем же `model × cluster × metric`, что и ноутбук со значимостью.

Основная логика сохранена такой же:

1. Рекурсивно ищем `before/after` файлы в `RESULTS_ROOT`.
2. Выбираем лучший per-user файл для каждой пары `model × cluster × condition`.
3. Собираем единую long-таблицу.
4. Делаем **paired matching по `user_id`**.
5. Для каждой метрики считаем:
   - `std_before` — стандартное отклонение метрики до эволюции;
   - `std_after` — стандартное отклонение метрики после эволюции;
   - `std_delta` — стандартное отклонение индивидуальных изменений `after - before`.

По умолчанию используется **выборочное стандартное отклонение**: `ddof=1`.

In [1]:
from __future__ import annotations

import importlib.util
import json
import re
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from tqdm.auto import tqdm
except Exception:  # noqa: BLE001
    tqdm = lambda x, **kwargs: x

In [2]:
# --- Path config ---
REPO_ROOT = Path('../').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

# Прямой путь к результатам EvoPrompt iter2.
# Если структура проекта другая, поправь только эту переменную.
RESULTS_ROOT = REPO_ROOT / 'results_experiments' / 'evoprompt_iter2'

# Куда сохранять таблицы
OUTPUT_DIR = REPO_ROOT / 'results_experiments' / 'evoprompt_iter2_std_tables'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RESULTS_ROOT:', RESULTS_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

if not RESULTS_ROOT.exists():
    raise FileNotFoundError(
        f'Не найдена папка RESULTS_ROOT={RESULTS_ROOT}. '
        'Проверь путь в ячейке Path config.'
    )

REPO_ROOT: D:\programming\GitHub\LLM-PersonaBench
RESULTS_ROOT: D:\programming\GitHub\LLM-PersonaBench\results_experiments\evoprompt_iter2
OUTPUT_DIR: D:\programming\GitHub\LLM-PersonaBench\results_experiments\evoprompt_iter2_std_tables


In [3]:
# --- Try to reuse existing metric code from personality_match.py ---
try:
    from src.utils.personality_match import compute_five_factor_metrics as compute_five_factor_metrics_pm
    PERSONALITY_MATCH_AVAILABLE = True
    PERSONALITY_MATCH_IMPORT_ERROR = None
except Exception as exc:  # noqa: BLE001
    compute_five_factor_metrics_pm = None
    PERSONALITY_MATCH_AVAILABLE = False
    PERSONALITY_MATCH_IMPORT_ERROR = repr(exc)

print('PERSONALITY_MATCH_AVAILABLE:', PERSONALITY_MATCH_AVAILABLE)
if not PERSONALITY_MATCH_AVAILABLE:
    print('Import error:', PERSONALITY_MATCH_IMPORT_ERROR)

PERSONALITY_MATCH_AVAILABLE: True


In [4]:
# --- Constants ---
USER_ID_CANDIDATES = ['user_id', 'participant_id', 'respondent_id', 'case', 'id']

# Четыре основные метрики из предыдущего ноутбука:
# - answer_similarity: similarity/accuracy по отдельным ответам
# - mae_35: MAE по итоговому профилю
# - trait_similarity: similarity по Big Five traits
# - facet_similarity: similarity по facets
METRIC_COLUMN_CANDIDATES = {
    'answer_similarity': ['answer_similarity', 'similarity', 'answer_accuracy', 'ans_accuracy'],
    'mae_35': ['mae_35', 'mae'],
    'trait_similarity': ['trait_similarity', 'mean_similarity_traits', 'similarity_traits'],
    'facet_similarity': ['facet_similarity', 'mean_similarity_facets', 'similarity_facets'],
}

METRICS = list(METRIC_COLUMN_CANDIDATES.keys())

METRIC_DIRECTIONS = {
    'answer_similarity': 'higher_is_better',
    'mae_35': 'lower_is_better',
    'trait_similarity': 'higher_is_better',
    'facet_similarity': 'higher_is_better',
}

# ddof=1 — выборочное стандартное отклонение.
# Если нужно population std, поставь STD_DDOF = 0.
STD_DDOF = 1

In [5]:
# --- Helpers: parsing and loading ---
def parse_cluster_from_segment(segment: str) -> str | None:
    segment_low = segment.lower()
    tokens = [t for t in re.split(r'[^a-z0-9]+', segment_low) if t]

    for i, tok in enumerate(tokens):
        if tok == 'cluster':
            cluster_ids: list[str] = []
            for nxt in tokens[i + 1:]:
                # Ограничение <=2 цифр помогает не захватывать таймстемпы
                if nxt.isdigit() and len(nxt) <= 2:
                    cluster_ids.append(str(int(nxt)))
                else:
                    break
            if cluster_ids:
                return 'cluster_' + '_'.join(cluster_ids)

    m_c = re.search(r'\bc[_\- ]?(\d{1,2})\b', segment_low)
    if m_c:
        return f"cluster_{int(m_c.group(1))}"

    m_cluster = re.search(r'cluster[_\- ]?(\d{1,2})\b', segment_low)
    if m_cluster:
        return f"cluster_{int(m_cluster.group(1))}"

    return None


def detect_cluster(path: Path) -> str | None:
    for part in reversed(path.parts):
        cluster = parse_cluster_from_segment(part)
        if cluster is not None:
            return cluster
    return None


def detect_condition(path: Path, df: pd.DataFrame | None = None) -> str | None:
    path_low = path.as_posix().lower()
    before_hit = (
        bool(re.search(r'(^|[^a-z])before([^a-z]|$)', path_low))
        or bool(re.search(r'(^|[^a-z])pre([^a-z]|$)', path_low))
    )
    after_hit = (
        bool(re.search(r'(^|[^a-z])after([^a-z]|$)', path_low))
        or bool(re.search(r'(^|[^a-z])post([^a-z]|$)', path_low))
    )

    if before_hit and not after_hit:
        return 'before'
    if after_hit and not before_hit:
        return 'after'

    if df is not None and 'condition' in df.columns:
        vals = set(df['condition'].astype(str).str.lower().str.strip().unique())
        if vals == {'before'}:
            return 'before'
        if vals == {'after'}:
            return 'after'

    return None


def load_tabular_file(path: Path, nrows: int | None = None) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == '.csv':
        return pd.read_csv(path, nrows=nrows)

    if suffix == '.jsonl':
        return pd.read_json(path, lines=True, nrows=nrows)

    if suffix == '.json':
        text = path.read_text(encoding='utf-8')
        obj = json.loads(text)

        if isinstance(obj, list):
            return pd.DataFrame(obj[:nrows] if nrows is not None else obj)

        if isinstance(obj, dict):
            list_of_dict_key = None
            for k, v in obj.items():
                if isinstance(v, list) and (not v or isinstance(v[0], dict)):
                    list_of_dict_key = k
                    break
            if list_of_dict_key is not None:
                records = obj[list_of_dict_key]
                return pd.DataFrame(records[:nrows] if nrows is not None else records)
            return pd.DataFrame([obj])

    raise ValueError(f'Unsupported file format: {path}')


def detect_user_id_col(df: pd.DataFrame) -> str | None:
    cols = [str(c) for c in df.columns]
    for candidate in USER_ID_CANDIDATES:
        if candidate in cols:
            return candidate
    return None


def to_float_series(df: pd.DataFrame, col: str) -> pd.Series:
    return pd.to_numeric(df[col], errors='coerce')


def maybe_compute_metrics_with_personality_match(df: pd.DataFrame) -> pd.DataFrame:
    """Fallback: если в файле нет profile-метрик, но есть real/sim profiles, пробуем пересчитать."""
    if compute_five_factor_metrics_pm is None:
        return df

    real_candidates = ['real_ocean', 'human_ocean', 'target_ocean', 'real_flat', 'human_profile']
    sim_candidates = ['simulated_ocean', 'model_ocean', 'pred_ocean', 'simulated_flat']

    real_col = next((c for c in real_candidates if c in df.columns), None)
    sim_col = next((c for c in sim_candidates if c in df.columns), None)
    if real_col is None or sim_col is None:
        return df

    missing_any = any(metric not in df.columns for metric in ['mae_35', 'mean_similarity_traits', 'mean_similarity_facets'])
    if not missing_any:
        return df

    computed = []
    for _, row in df.iterrows():
        real_flat = row.get(real_col)
        sim_flat = row.get(sim_col)
        if not isinstance(real_flat, dict) or not isinstance(sim_flat, dict):
            computed.append(None)
            continue
        try:
            metrics = compute_five_factor_metrics_pm(real_flat, sim_flat)
        except Exception:  # noqa: BLE001
            metrics = None
        computed.append(metrics)

    computed_series = pd.Series(computed, index=df.index)
    if 'mae_35' not in df.columns:
        df['mae_35'] = computed_series.apply(lambda x: x.get('mae_35') if isinstance(x, dict) else np.nan)
    if 'mean_similarity_traits' not in df.columns:
        df['mean_similarity_traits'] = computed_series.apply(
            lambda x: x.get('mean_similarity_traits') if isinstance(x, dict) else np.nan
        )
    if 'mean_similarity_facets' not in df.columns:
        df['mean_similarity_facets'] = computed_series.apply(
            lambda x: x.get('mean_similarity_facets') if isinstance(x, dict) else np.nan
        )

    return df


def sample_std(values: np.ndarray | pd.Series, ddof: int = STD_DDOF) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        return np.nan

    # Для ddof=1 нужно минимум 2 наблюдения.
    if arr.size <= ddof:
        return np.nan

    return float(np.std(arr, ddof=ddof))

In [6]:
# --- Discovery / selection ---
def discover_candidate_files(results_root: Path) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    supported = {'.csv', '.json', '.jsonl'}

    for path in tqdm(sorted(results_root.rglob('*')), desc='Discovering files'):
        if not path.is_file() or path.suffix.lower() not in supported:
            continue

        condition_guess = detect_condition(path)
        if condition_guess is None:
            continue

        rel = path.relative_to(results_root)
        model = rel.parts[0] if len(rel.parts) >= 2 else 'unknown'
        cluster = detect_cluster(path)

        head_df = pd.DataFrame()
        load_error = None
        try:
            head_df = load_tabular_file(path, nrows=5)
        except Exception as exc:  # noqa: BLE001
            load_error = repr(exc)

        cols = [str(c) for c in head_df.columns]
        metric_cols_found = sorted({
            c for c in cols for options in METRIC_COLUMN_CANDIDATES.values() if c in options
        })
        has_user_id = detect_user_id_col(head_df) is not None
        has_answer_cols = any(re.fullmatch(r'i\d+', c.lower()) for c in cols)

        score = 0
        if metric_cols_found:
            score += 100
        if 'participants' in path.name.lower():
            score += 50
        if has_answer_cols:
            score += 20
        if has_user_id:
            score += 10
        if 'result_log' in path.as_posix().lower():
            score -= 100
        if head_df.shape[0] <= 1:
            score -= 20

        records.append(
            {
                'model': model,
                'cluster': cluster,
                'condition': condition_guess,
                'path': str(path),
                'suffix': path.suffix.lower(),
                'file_name': path.name,
                'sample_rows': int(head_df.shape[0]),
                'sample_cols': int(head_df.shape[1]),
                'columns_sample': ', '.join(cols[:20]),
                'has_user_id': bool(has_user_id),
                'has_metric_cols': bool(len(metric_cols_found) > 0),
                'metric_cols_found': ', '.join(metric_cols_found),
                'has_answer_cols': bool(has_answer_cols),
                'score': score,
                'mtime': path.stat().st_mtime,
                'load_error': load_error,
            }
        )

    if not records:
        return pd.DataFrame(columns=['model', 'cluster', 'condition', 'path'])

    return pd.DataFrame(records)


def select_best_files(candidates_df: pd.DataFrame) -> pd.DataFrame:
    if candidates_df.empty:
        return candidates_df

    eligible = candidates_df[candidates_df['cluster'].notna()].copy()
    eligible = eligible.sort_values(
        ['model', 'cluster', 'condition', 'score', 'mtime', 'path'],
        ascending=[True, True, True, False, False, True],
    )

    best = eligible.groupby(['model', 'cluster', 'condition'], as_index=False).head(1).copy()
    return best


candidates_df = discover_candidate_files(RESULTS_ROOT)
print('Найдено candidate before/after файлов:', len(candidates_df))

display_cols = [
    'model', 'cluster', 'condition', 'file_name', 'suffix',
    'sample_rows', 'sample_cols', 'has_user_id', 'has_metric_cols',
    'metric_cols_found', 'has_answer_cols', 'score', 'path', 'load_error'
]
display(candidates_df.sort_values(['model', 'cluster', 'condition', 'score'], ascending=[True, True, True, False])[display_cols])

selected_df = select_best_files(candidates_df)
print()
print('Выбрано per-user файлов для анализа:', len(selected_df))
display(selected_df.sort_values(['model', 'cluster', 'condition'])[display_cols])

# Проверка, что на каждый model×cluster есть both before+after
combo_check = selected_df.groupby(['model', 'cluster'])['condition'].agg(lambda s: sorted(set(s))).reset_index()
combo_check['has_before'] = combo_check['condition'].apply(lambda x: 'before' in x)
combo_check['has_after'] = combo_check['condition'].apply(lambda x: 'after' in x)
combo_check['ok_pair'] = combo_check['has_before'] & combo_check['has_after']

print()
print('Проверка наличия before/after на model×cluster:')
display(combo_check)

missing_pair_df = combo_check[~combo_check['ok_pair']].copy()
if not missing_pair_df.empty:
    print('WARNING: найдены model×cluster без полной пары before/after')
    display(missing_pair_df)

Discovering files:   0%|          | 0/247 [00:00<?, ?it/s]

Найдено candidate before/after файлов: 84


,model,cluster,condition,file_name,suffix,sample_rows,sample_cols,has_user_id,has_metric_cols,metric_cols_found,has_answer_cols,score,path,load_error
1,gigchat3,cluster_0,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
0,gigchat3,cluster_0,after,after_optimization_test_answers.csv,.csv,5,121,True,False,,True,30,D:\programming\GitHub\LLM-PersonaBench\results...,None
3,gigchat3,cluster_0,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
2,gigchat3,cluster_0,before,before_optimization_test_answers.csv,.csv,5,121,True,False,,True,30,D:\programming\GitHub\LLM-PersonaBench\results...,None
5,gigchat3,cluster_1,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,qwen3,cluster_2,before,before_optimization_test_answers.csv,.csv,5,121,True,False,,True,30,D:\programming\GitHub\LLM-PersonaBench\results...,None
81,qwen3,cluster_3,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
80,qwen3,cluster_3,after,after_optimization_test_answers.csv,.csv,5,121,True,False,,True,30,D:\programming\GitHub\LLM-PersonaBench\results...,None
83,qwen3,cluster_3,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None



Выбрано per-user файлов для анализа: 40


,model,cluster,condition,file_name,suffix,sample_rows,sample_cols,has_user_id,has_metric_cols,metric_cols_found,has_answer_cols,score,path,load_error
1,gigchat3,cluster_0,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
3,gigchat3,cluster_0,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
5,gigchat3,cluster_1,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
7,gigchat3,cluster_1,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
9,gigchat3,cluster_2,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
11,gigchat3,cluster_2,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
13,gigchat3,cluster_3,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
15,gigchat3,cluster_3,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
17,gpt4_mini,cluster_0,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
19,gpt4_mini,cluster_0,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None



Проверка наличия before/after на model×cluster:


,model,cluster,condition,has_before,has_after,ok_pair
0,gigchat3,cluster_0,"[after, before]",True,True,True
1,gigchat3,cluster_1,"[after, before]",True,True,True
2,gigchat3,cluster_2,"[after, before]",True,True,True
3,gigchat3,cluster_3,"[after, before]",True,True,True
4,gpt4_mini,cluster_0,"[after, before]",True,True,True
5,gpt4_mini,cluster_1,"[after, before]",True,True,True
6,gpt4_mini,cluster_2,"[after, before]",True,True,True
7,gpt4_mini,cluster_3,"[after, before]",True,True,True
8,gpt4_nano,cluster_0,"[after, before]",True,True,True
9,gpt4_nano,cluster_1,"[after, before]",True,True,True


In [7]:
# --- Load selected files into unified long format ---
def build_long_metrics_df(selected_df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    warnings_list: list[str] = []
    frames: list[pd.DataFrame] = []

    for row in tqdm(selected_df.itertuples(index=False), total=len(selected_df), desc='Loading selected files'):
        path = Path(row.path)
        try:
            df = load_tabular_file(path, nrows=None)
        except Exception as exc:  # noqa: BLE001
            warnings_list.append(f'Failed to load {path}: {exc}')
            continue

        df = maybe_compute_metrics_with_personality_match(df)

        user_col = detect_user_id_col(df)
        if user_col is None:
            warnings_list.append(f'No user id column in {path}')
            continue

        out = pd.DataFrame(
            {
                'model': row.model,
                'cluster': row.cluster,
                'condition': row.condition,
                'user_id': df[user_col].astype(str),
                'source_file': str(path),
            }
        )

        metric_missing_for_file = []
        for metric, col_candidates in METRIC_COLUMN_CANDIDATES.items():
            chosen_col = next((c for c in col_candidates if c in df.columns), None)
            if chosen_col is not None:
                out[metric] = to_float_series(df, chosen_col)
            elif metric == 'answer_similarity' and 'avg_diff' in df.columns:
                # fallback, если есть avg_diff по шкале 1..5
                out[metric] = 1.0 - to_float_series(df, 'avg_diff') / 4.0
            else:
                out[metric] = np.nan
                metric_missing_for_file.append(metric)

        if metric_missing_for_file:
            warnings_list.append(
                f"Missing metrics in {path.name}: {', '.join(metric_missing_for_file)}"
            )

        frames.append(out)

    if not frames:
        return pd.DataFrame(columns=['model', 'cluster', 'condition', 'user_id', *METRICS]), warnings_list

    long_df = pd.concat(frames, ignore_index=True)

    # На случай дублей user_id внутри одного source condition: агрегируем mean
    long_df = (
        long_df.groupby(['model', 'cluster', 'condition', 'user_id'], as_index=False)[METRICS]
        .mean()
    )

    return long_df, warnings_list


long_df, load_warnings = build_long_metrics_df(selected_df)
print('Размер long_df:', long_df.shape)
display(long_df.head())

users_before_pairing_df = (
    long_df.groupby(['model', 'cluster', 'condition'], as_index=False)['user_id']
    .nunique()
    .rename(columns={'user_id': 'n_users'})
    .sort_values(['model', 'cluster', 'condition'])
)

print()
print('Количество пользователей до pairing (model×cluster×condition):')
display(users_before_pairing_df)

if load_warnings:
    print()
    print('WARNING: проблемы при загрузке/метриках:')
    display(pd.DataFrame({'warning': load_warnings}))

Loading selected files:   0%|          | 0/40 [00:00<?, ?it/s]

Размер long_df: (1600, 8)


,model,cluster,condition,user_id,answer_similarity,mae_35,trait_similarity,facet_similarity
0,gigchat3,cluster_0,after,529,0.695833,24.611642,0.776161,0.750171
1,gigchat3,cluster_0,after,540,0.689583,27.945727,0.797689,0.707685
2,gigchat3,cluster_0,after,542,0.718750,28.443267,0.729411,0.713260
3,gigchat3,cluster_0,after,547,0.787500,20.677137,0.803363,0.791540
4,gigchat3,cluster_0,after,570,0.697917,23.371709,0.810774,0.758868



Количество пользователей до pairing (model×cluster×condition):


,model,cluster,condition,n_users
0,gigchat3,cluster_0,after,40
1,gigchat3,cluster_0,before,40
2,gigchat3,cluster_1,after,40
3,gigchat3,cluster_1,before,40
4,gigchat3,cluster_2,after,40
5,gigchat3,cluster_2,before,40
6,gigchat3,cluster_3,after,40
7,gigchat3,cluster_3,before,40
8,gpt4_mini,cluster_0,after,40
9,gpt4_mini,cluster_0,before,40


In [8]:
# --- Build paired wide dataframe ---
def build_pairing(long_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    diagnostics = []
    paired_frames = []

    for (model, cluster), group in long_df.groupby(['model', 'cluster']):
        before = group[group['condition'] == 'before'].set_index('user_id')
        after = group[group['condition'] == 'after'].set_index('user_id')

        b_users = set(before.index)
        a_users = set(after.index)
        paired_users = sorted(b_users & a_users)

        diagnostics.append(
            {
                'model': model,
                'cluster': cluster,
                'n_before_users': len(b_users),
                'n_after_users': len(a_users),
                'n_paired_users': len(paired_users),
                'n_excluded_before_only': len(b_users - a_users),
                'n_excluded_after_only': len(a_users - b_users),
            }
        )

        if paired_users:
            merged = before.loc[paired_users, METRICS].add_suffix('_before').join(
                after.loc[paired_users, METRICS].add_suffix('_after'),
                how='inner',
            )
            merged = merged.reset_index().rename(columns={'index': 'user_id'})
            merged.insert(0, 'cluster', cluster)
            merged.insert(0, 'model', model)
            paired_frames.append(merged)

    pairing_diag_df = pd.DataFrame(diagnostics).sort_values(['model', 'cluster']).reset_index(drop=True)

    if paired_frames:
        wide_df = pd.concat(paired_frames, ignore_index=True)
    else:
        wide_cols = ['model', 'cluster', 'user_id']
        for metric in METRICS:
            wide_cols.extend([f'{metric}_before', f'{metric}_after'])
        wide_df = pd.DataFrame(columns=wide_cols)

    return wide_df, pairing_diag_df


wide_df, pairing_diag_df = build_pairing(long_df)

print('Размер wide_df (paired users):', wide_df.shape)
print()
print('Диагностика pairing по model×cluster:')
display(pairing_diag_df)

unpaired_df = pairing_diag_df[
    (pairing_diag_df['n_excluded_before_only'] > 0)
    | (pairing_diag_df['n_excluded_after_only'] > 0)
].copy()

if not unpaired_df.empty:
    print('WARNING: есть непарные записи, исключённые из paired анализа')
    display(unpaired_df)

Размер wide_df (paired users): (800, 11)

Диагностика pairing по model×cluster:


,model,cluster,n_before_users,n_after_users,n_paired_users,n_excluded_before_only,n_excluded_after_only
0,gigchat3,cluster_0,40,40,40,0,0
1,gigchat3,cluster_1,40,40,40,0,0
2,gigchat3,cluster_2,40,40,40,0,0
3,gigchat3,cluster_3,40,40,40,0,0
4,gpt4_mini,cluster_0,40,40,40,0,0
5,gpt4_mini,cluster_1,40,40,40,0,0
6,gpt4_mini,cluster_2,40,40,40,0,0
7,gpt4_mini,cluster_3,40,40,40,0,0
8,gpt4_nano,cluster_0,40,40,40,0,0
9,gpt4_nano,cluster_1,40,40,40,0,0


In [9]:
# --- Standard deviation summary ---
def summarize_std(
    wide_df: pd.DataFrame,
    ddof: int = STD_DDOF,
) -> tuple[pd.DataFrame, list[str]]:
    rows = []
    warnings_list = []

    for (model, cluster), chunk in wide_df.groupby(['model', 'cluster']):
        for metric in METRICS:
            before_col = f'{metric}_before'
            after_col = f'{metric}_after'

            before_vals = pd.to_numeric(chunk[before_col], errors='coerce').to_numpy(dtype=float)
            after_vals = pd.to_numeric(chunk[after_col], errors='coerce').to_numpy(dtype=float)

            # Важно: std_before/std_after/std_delta считаем на одном и том же наборе пользователей,
            # где есть и before, и after для конкретной метрики.
            finite_mask = np.isfinite(before_vals) & np.isfinite(after_vals)

            before_complete = before_vals[finite_mask]
            after_complete = after_vals[finite_mask]
            delta = after_complete - before_complete

            n_total_paired_users = len(chunk)
            n_valid_metric_pairs = int(finite_mask.sum())
            n_missing_metric_pairs = int(n_total_paired_users - n_valid_metric_pairs)

            if n_missing_metric_pairs > 0:
                warnings_list.append(
                    f"{model}/{cluster}/{metric}: "
                    f"{n_missing_metric_pairs} paired users excluded because "
                    f"before or after metric is missing"
                )

            std_delta = sample_std(delta, ddof=ddof)

            rows.append(
                {
                    'model': model,
                    'cluster': cluster,
                    'metric': metric,
                    'direction': METRIC_DIRECTIONS.get(metric),
                    'n_paired_users_total': n_total_paired_users,
                    'n_users': n_valid_metric_pairs,
                    'n_missing_metric_pairs': n_missing_metric_pairs,

                    # Средние оставлены как диагностические поля,
                    # чтобы было видно, вокруг какого уровня считается разброс.
                    'mean_before': float(np.mean(before_complete)) if n_valid_metric_pairs > 0 else np.nan,
                    'mean_after': float(np.mean(after_complete)) if n_valid_metric_pairs > 0 else np.nan,
                    'mean_delta': float(np.mean(delta)) if n_valid_metric_pairs > 0 else np.nan,

                    # Главное: стандартные отклонения.
                    'std_before': sample_std(before_complete, ddof=ddof),
                    'std_after': sample_std(after_complete, ddof=ddof),
                    'std_delta': std_delta,

                    # Дополнительно удобно для быстрой интерпретации.
                    'sem_delta': (
                        std_delta / np.sqrt(n_valid_metric_pairs)
                        if n_valid_metric_pairs > ddof and np.isfinite(std_delta) else np.nan
                    ),
                    'min_delta': float(np.min(delta)) if n_valid_metric_pairs > 0 else np.nan,
                    'max_delta': float(np.max(delta)) if n_valid_metric_pairs > 0 else np.nan,
                    'std_ddof': ddof,
                }
            )

    std_summary_df = (
        pd.DataFrame(rows)
        .sort_values(['model', 'cluster', 'metric'])
        .reset_index(drop=True)
    )

    return std_summary_df, warnings_list


std_summary_df, std_warnings = summarize_std(wide_df, ddof=STD_DDOF)

print('Размер std_summary_df:', std_summary_df.shape)
display(std_summary_df)

if std_warnings:
    print('WARNING: std warnings')
    display(pd.DataFrame({'warning': std_warnings}))

Размер std_summary_df: (80, 17)


,model,cluster,metric,direction,n_paired_users_total,n_users,n_missing_metric_pairs,mean_before,mean_after,mean_delta,std_before,std_after,std_delta,sem_delta,min_delta,max_delta,std_ddof
0,gigchat3,cluster_0,answer_similarity,higher_is_better,40,38,2,0.664912,0.712901,0.047988,0.053008,0.044706,0.041647,0.006756,-0.047917,0.129167,1
1,gigchat3,cluster_0,facet_similarity,higher_is_better,40,37,3,0.747101,0.730418,-0.016683,0.030980,0.036500,0.035883,0.005899,-0.103171,0.059593,1
2,gigchat3,cluster_0,mae_35,lower_is_better,40,37,3,24.710970,26.215783,1.504813,3.197652,3.506136,3.487813,0.573393,-6.632290,10.138413,1
3,gigchat3,cluster_0,trait_similarity,higher_is_better,40,37,3,0.787629,0.782389,-0.005240,0.058326,0.051646,0.053285,0.008760,-0.117171,0.106701,1
4,gigchat3,cluster_1,answer_similarity,higher_is_better,40,35,5,0.675417,0.689643,0.014226,0.038151,0.043635,0.034557,0.005841,-0.058333,0.083333,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,qwen3,cluster_2,trait_similarity,higher_is_better,40,40,0,0.761994,0.734682,-0.027312,0.073353,0.084964,0.070578,0.011159,-0.198140,0.102254,1
76,qwen3,cluster_3,answer_similarity,higher_is_better,40,40,0,0.642552,0.642083,-0.000469,0.040811,0.047962,0.025605,0.004048,-0.056250,0.070833,1
77,qwen3,cluster_3,facet_similarity,higher_is_better,40,40,0,0.690351,0.685673,-0.004677,0.062487,0.066761,0.029715,0.004698,-0.078922,0.066684,1
78,qwen3,cluster_3,mae_35,lower_is_better,40,40,0,30.830760,31.239484,0.408724,6.662709,6.990073,3.632498,0.574348,-6.462151,14.926595,1


,warning
0,gigchat3/cluster_0/answer_similarity: 2 paired...
1,gigchat3/cluster_0/mae_35: 3 paired users excl...
2,gigchat3/cluster_0/trait_similarity: 3 paired ...
3,gigchat3/cluster_0/facet_similarity: 3 paired ...
4,gigchat3/cluster_1/answer_similarity: 5 paired...
5,gigchat3/cluster_1/mae_35: 5 paired users excl...
6,gigchat3/cluster_1/trait_similarity: 5 paired ...
7,gigchat3/cluster_1/facet_similarity: 5 paired ...
8,gigchat3/cluster_2/answer_similarity: 1 paired...
9,gigchat3/cluster_2/mae_35: 1 paired users excl...


In [10]:
# --- Pivot tables ---
def build_std_outputs(
    wide_df: pd.DataFrame,
    std_summary_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if std_summary_df.empty:
        empty_base = pd.DataFrame(columns=['model', 'cluster'])
        paired_user_deltas_df = pd.DataFrame(columns=['model', 'cluster', 'user_id'])
        return empty_base, empty_base.copy(), empty_base.copy(), paired_user_deltas_df

    # 1) Главная компактная таблица: std_delta по всем четырём метрикам
    std_delta_pivot_df = std_summary_df.pivot_table(
        index=['model', 'cluster'],
        columns='metric',
        values='std_delta',
        aggfunc='first',
    ).reset_index()

    # 2) Таблица std_before/std_after/std_delta в широком формате
    std_full_pivot_df = std_summary_df.pivot_table(
        index=['model', 'cluster'],
        columns='metric',
        values=['std_before', 'std_after', 'std_delta'],
        aggfunc='first',
    )
    std_full_pivot_df.columns = [f'{stat}__{metric}' for stat, metric in std_full_pivot_df.columns]
    std_full_pivot_df = std_full_pivot_df.reset_index()

    # 3) Диагностическая таблица со средними delta
    mean_delta_pivot_df = std_summary_df.pivot_table(
        index=['model', 'cluster'],
        columns='metric',
        values='mean_delta',
        aggfunc='first',
    ).reset_index()

    # 4) User-level deltas, чтобы при необходимости проверить расчёты руками
    paired_user_deltas_df = wide_df[['model', 'cluster', 'user_id']].copy() if not wide_df.empty else pd.DataFrame(
        columns=['model', 'cluster', 'user_id']
    )
    for metric in METRICS:
        if not wide_df.empty:
            paired_user_deltas_df[f'{metric}_delta'] = (
                pd.to_numeric(wide_df[f'{metric}_after'], errors='coerce')
                - pd.to_numeric(wide_df[f'{metric}_before'], errors='coerce')
            )
        else:
            paired_user_deltas_df[f'{metric}_delta'] = []

    return std_delta_pivot_df, std_full_pivot_df, mean_delta_pivot_df, paired_user_deltas_df


std_delta_pivot_df, std_full_pivot_df, mean_delta_pivot_df, paired_user_deltas_df = build_std_outputs(
    wide_df=wide_df,
    std_summary_df=std_summary_df,
)

print('A) std_summary_long')
display(std_summary_df)

print('B) compact pivot: std_delta')
display(std_delta_pivot_df)

print('C) full pivot: std_before/std_after/std_delta')
display(std_full_pivot_df)

print('D) mean_delta pivot, only for diagnostics')
display(mean_delta_pivot_df)

A) std_summary_long


,model,cluster,metric,direction,n_paired_users_total,n_users,n_missing_metric_pairs,mean_before,mean_after,mean_delta,std_before,std_after,std_delta,sem_delta,min_delta,max_delta,std_ddof
0,gigchat3,cluster_0,answer_similarity,higher_is_better,40,38,2,0.664912,0.712901,0.047988,0.053008,0.044706,0.041647,0.006756,-0.047917,0.129167,1
1,gigchat3,cluster_0,facet_similarity,higher_is_better,40,37,3,0.747101,0.730418,-0.016683,0.030980,0.036500,0.035883,0.005899,-0.103171,0.059593,1
2,gigchat3,cluster_0,mae_35,lower_is_better,40,37,3,24.710970,26.215783,1.504813,3.197652,3.506136,3.487813,0.573393,-6.632290,10.138413,1
3,gigchat3,cluster_0,trait_similarity,higher_is_better,40,37,3,0.787629,0.782389,-0.005240,0.058326,0.051646,0.053285,0.008760,-0.117171,0.106701,1
4,gigchat3,cluster_1,answer_similarity,higher_is_better,40,35,5,0.675417,0.689643,0.014226,0.038151,0.043635,0.034557,0.005841,-0.058333,0.083333,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,qwen3,cluster_2,trait_similarity,higher_is_better,40,40,0,0.761994,0.734682,-0.027312,0.073353,0.084964,0.070578,0.011159,-0.198140,0.102254,1
76,qwen3,cluster_3,answer_similarity,higher_is_better,40,40,0,0.642552,0.642083,-0.000469,0.040811,0.047962,0.025605,0.004048,-0.056250,0.070833,1
77,qwen3,cluster_3,facet_similarity,higher_is_better,40,40,0,0.690351,0.685673,-0.004677,0.062487,0.066761,0.029715,0.004698,-0.078922,0.066684,1
78,qwen3,cluster_3,mae_35,lower_is_better,40,40,0,30.830760,31.239484,0.408724,6.662709,6.990073,3.632498,0.574348,-6.462151,14.926595,1


B) compact pivot: std_delta


metric,model,cluster,answer_similarity,facet_similarity,mae_35,trait_similarity
0,gigchat3,cluster_0,0.041647,0.035883,3.487813,0.053285
1,gigchat3,cluster_1,0.034557,0.024581,2.369066,0.040940
2,gigchat3,cluster_2,0.022025,0.025963,2.677161,0.051375
3,gigchat3,cluster_3,0.077319,0.058048,5.478044,0.070807
4,gpt4_mini,cluster_0,0.015539,0.021494,2.181037,0.032335
5,gpt4_mini,cluster_1,0.009199,0.009722,0.992292,0.016689
6,gpt4_mini,cluster_2,0.023716,0.045759,5.179395,0.107536
7,gpt4_mini,cluster_3,0.017333,0.019833,1.983517,0.041995
8,gpt4_nano,cluster_0,0.010210,0.015519,1.590142,0.025069
9,gpt4_nano,cluster_1,0.011028,0.014674,1.408557,0.026481


C) full pivot: std_before/std_after/std_delta


,model,cluster,std_after__answer_similarity,std_after__facet_similarity,std_after__mae_35,std_after__trait_similarity,std_before__answer_similarity,std_before__facet_similarity,std_before__mae_35,std_before__trait_similarity,std_delta__answer_similarity,std_delta__facet_similarity,std_delta__mae_35,std_delta__trait_similarity
0,gigchat3,cluster_0,0.044706,0.036500,3.506136,0.051646,0.053008,0.030980,3.197652,0.058326,0.041647,0.035883,3.487813,0.053285
1,gigchat3,cluster_1,0.043635,0.047182,4.900768,0.075454,0.038151,0.042858,4.416701,0.069864,0.034557,0.024581,2.369066,0.040940
2,gigchat3,cluster_2,0.044304,0.046888,4.616714,0.063286,0.046756,0.052305,5.039571,0.066230,0.022025,0.025963,2.677161,0.051375
3,gigchat3,cluster_3,0.053467,0.064316,6.716557,0.108312,0.054938,0.065765,6.527394,0.104050,0.077319,0.058048,5.478044,0.070807
4,gpt4_mini,cluster_0,0.037173,0.040118,4.115280,0.061364,0.033740,0.038756,3.999253,0.062683,0.015539,0.021494,2.181037,0.032335
5,gpt4_mini,cluster_1,0.031919,0.038207,3.950480,0.056693,0.033672,0.038244,3.975781,0.056621,0.009199,0.009722,0.992292,0.016689
6,gpt4_mini,cluster_2,0.047589,0.038291,3.817774,0.058189,0.049463,0.047918,4.948244,0.079162,0.023716,0.045759,5.179395,0.107536
7,gpt4_mini,cluster_3,0.038631,0.063124,6.044800,0.064819,0.035661,0.064106,6.329975,0.082005,0.017333,0.019833,1.983517,0.041995
8,gpt4_nano,cluster_0,0.030709,0.042061,4.296154,0.060327,0.029876,0.041459,4.187813,0.055861,0.010210,0.015519,1.590142,0.025069
9,gpt4_nano,cluster_1,0.030582,0.035377,3.342343,0.039023,0.029774,0.030231,3.091019,0.051528,0.011028,0.014674,1.408557,0.026481


D) mean_delta pivot, only for diagnostics


metric,model,cluster,answer_similarity,facet_similarity,mae_35,trait_similarity
0,gigchat3,cluster_0,0.047988,-0.016683,1.504813,-0.005240
1,gigchat3,cluster_1,0.014226,-0.007622,0.434511,0.015315
2,gigchat3,cluster_2,-0.002137,-0.002865,0.389606,-0.010081
3,gigchat3,cluster_3,0.035000,0.016327,-1.322515,-0.005386
4,gpt4_mini,cluster_0,0.002969,-0.001861,0.321770,-0.011355
5,gpt4_mini,cluster_1,0.001667,-0.000790,0.055086,0.000883
6,gpt4_mini,cluster_2,0.022813,0.027336,-2.150820,-0.013461
7,gpt4_mini,cluster_3,0.004687,-0.010536,1.178567,-0.019285
8,gpt4_nano,cluster_0,0.000156,-0.000382,0.009167,0.001652
9,gpt4_nano,cluster_1,0.004844,0.001048,-0.218319,0.008997


In [11]:
# --- Optional formatting for paper/report view ---
def format_number(x: float, decimals: int = 4) -> str:
    if not np.isfinite(x):
        return 'n/a'
    return f'{x:.{decimals}f}'


def make_paper_like_std_table(std_summary_df: pd.DataFrame) -> pd.DataFrame:
    """Компактная таблица: model, cluster, и std_delta по каждой метрике."""
    if std_summary_df.empty:
        return pd.DataFrame(columns=['model', 'cluster', *METRICS])

    table = std_summary_df.pivot_table(
        index=['model', 'cluster'],
        columns='metric',
        values='std_delta',
        aggfunc='first',
    ).reset_index()

    # Форматируем только для визуального просмотра.
    # В Excel/CSV сохраняется и raw-версия, и formatted-версия.
    formatted = table.copy()
    for metric in METRICS:
        if metric in formatted.columns:
            decimals = 3 if metric == 'mae_35' else 4
            formatted[metric] = formatted[metric].apply(lambda x: format_number(x, decimals=decimals))

    return formatted


std_delta_pivot_formatted_df = make_paper_like_std_table(std_summary_df)

print('Formatted std_delta table:')
display(std_delta_pivot_formatted_df)

Formatted std_delta table:


metric,model,cluster,answer_similarity,facet_similarity,mae_35,trait_similarity
0,gigchat3,cluster_0,0.0416,0.0359,3.488,0.0533
1,gigchat3,cluster_1,0.0346,0.0246,2.369,0.0409
2,gigchat3,cluster_2,0.0220,0.0260,2.677,0.0514
3,gigchat3,cluster_3,0.0773,0.0580,5.478,0.0708
4,gpt4_mini,cluster_0,0.0155,0.0215,2.181,0.0323
5,gpt4_mini,cluster_1,0.0092,0.0097,0.992,0.0167
6,gpt4_mini,cluster_2,0.0237,0.0458,5.179,0.1075
7,gpt4_mini,cluster_3,0.0173,0.0198,1.984,0.0420
8,gpt4_nano,cluster_0,0.0102,0.0155,1.590,0.0251
9,gpt4_nano,cluster_1,0.0110,0.0147,1.409,0.0265


In [12]:
# --- Save outputs ---
summary_csv = OUTPUT_DIR / 'std_summary_long.csv'
summary_xlsx = OUTPUT_DIR / 'std_summary_tables.xlsx'
std_delta_pivot_csv = OUTPUT_DIR / 'std_delta_pivot.csv'
std_full_pivot_csv = OUTPUT_DIR / 'std_full_pivot.csv'
paired_csv = OUTPUT_DIR / 'paired_user_deltas_for_std.csv'

std_summary_df.to_csv(summary_csv, index=False)
std_delta_pivot_df.to_csv(std_delta_pivot_csv, index=False)
std_full_pivot_df.to_csv(std_full_pivot_csv, index=False)
paired_user_deltas_df.to_csv(paired_csv, index=False)

excel_engine = None
if importlib.util.find_spec('openpyxl') is not None:
    excel_engine = 'openpyxl'
elif importlib.util.find_spec('xlsxwriter') is not None:
    excel_engine = 'xlsxwriter'

if excel_engine is not None:
    with pd.ExcelWriter(summary_xlsx, engine=excel_engine) as writer:
        std_summary_df.to_excel(writer, sheet_name='std_summary_long', index=False)
        std_delta_pivot_df.to_excel(writer, sheet_name='std_delta_pivot_raw', index=False)
        std_delta_pivot_formatted_df.to_excel(writer, sheet_name='std_delta_pivot_fmt', index=False)
        std_full_pivot_df.to_excel(writer, sheet_name='std_full_pivot', index=False)
        mean_delta_pivot_df.to_excel(writer, sheet_name='mean_delta_diagnostic', index=False)
        pairing_diag_df.to_excel(writer, sheet_name='pairing_diagnostics', index=False)
        users_before_pairing_df.to_excel(writer, sheet_name='users_before_pairing', index=False)

print('Saved:', summary_csv)
print('Saved:', std_delta_pivot_csv)
print('Saved:', std_full_pivot_csv)
print('Saved:', paired_csv)

if excel_engine is not None:
    print('Saved:', summary_xlsx)
else:
    print('WARNING: openpyxl/xlsxwriter не найден; xlsx-файл не сохранён в этой среде.')

Saved: D:\programming\GitHub\LLM-PersonaBench\results_experiments\evoprompt_iter2_std_tables\std_summary_long.csv
Saved: D:\programming\GitHub\LLM-PersonaBench\results_experiments\evoprompt_iter2_std_tables\std_delta_pivot.csv
Saved: D:\programming\GitHub\LLM-PersonaBench\results_experiments\evoprompt_iter2_std_tables\std_full_pivot.csv
Saved: D:\programming\GitHub\LLM-PersonaBench\results_experiments\evoprompt_iter2_std_tables\paired_user_deltas_for_std.csv
Saved: D:\programming\GitHub\LLM-PersonaBench\results_experiments\evoprompt_iter2_std_tables\std_summary_tables.xlsx


In [13]:
# --- Consolidated diagnostics ---
all_warnings = []

if not missing_pair_df.empty:
    all_warnings.append('Есть model×cluster без полной пары before/after (см. таблицу выше).')

if load_warnings:
    all_warnings.extend(load_warnings)

if std_warnings:
    all_warnings.extend(std_warnings)

if not PERSONALITY_MATCH_AVAILABLE:
    all_warnings.append(
        'Не удалось импортировать src.utils.personality_match. '
        'Для текущих participants-файлов это обычно не блокирует расчёт, если метрики уже сохранены per-user.'
    )

print('Итоговые размеры:')
print(' - candidates_df:', candidates_df.shape)
print(' - selected_df:', selected_df.shape)
print(' - long_df:', long_df.shape)
print(' - wide_df (paired):', wide_df.shape)
print(' - std_summary_df:', std_summary_df.shape)
print(' - std_delta_pivot_df:', std_delta_pivot_df.shape)
print(' - std_full_pivot_df:', std_full_pivot_df.shape)
print(' - paired_user_deltas_df:', paired_user_deltas_df.shape)

if all_warnings:
    print()
    print('Warnings:')
    display(pd.DataFrame({'warning': all_warnings}))
else:
    print()
    print('Warnings: none')

Итоговые размеры:
 - candidates_df: (84, 16)
 - selected_df: (40, 16)
 - long_df: (1600, 8)
 - wide_df (paired): (800, 11)
 - std_summary_df: (80, 17)
 - std_delta_pivot_df: (20, 6)
 - std_full_pivot_df: (20, 14)
 - paired_user_deltas_df: (800, 7)

Warnings:


,warning
0,gigchat3/cluster_0/answer_similarity: 2 paired...
1,gigchat3/cluster_0/mae_35: 3 paired users excl...
2,gigchat3/cluster_0/trait_similarity: 3 paired ...
3,gigchat3/cluster_0/facet_similarity: 3 paired ...
4,gigchat3/cluster_1/answer_similarity: 5 paired...
5,gigchat3/cluster_1/mae_35: 5 paired users excl...
6,gigchat3/cluster_1/trait_similarity: 5 paired ...
7,gigchat3/cluster_1/facet_similarity: 5 paired ...
8,gigchat3/cluster_2/answer_similarity: 1 paired...
9,gigchat3/cluster_2/mae_35: 1 paired users excl...
